In [2]:
import nltk
from nltk.stem import WordNetLemmatizer
lemmatizer = WordNetLemmatizer()
import json
import pickle

import numpy as np
from keras.models import Sequential
from keras.layers import Dense, Activation, Dropout
from keras.optimizers import SGD
import random

In [3]:
words = []
classes = []
documents = []
ignore_words = ['?', '!']
data_file = open('gf.json').read()
intents = json.loads(data_file)


In [4]:
for intent in intents['intents']:
    for pattern in intent['patterns']:
        # tokenize each word
        w = nltk.word_tokenize(pattern)
        words.extend(w)
        # add documents in the corpus
        documents.append((w, intent['tag']))

        # add to our classes list
        if intent['tag'] not in classes:
            classes.append(intent['tag'])



In [5]:

# lemmatize and lower each word and remove duplicates
words = [lemmatizer.lemmatize(w.lower()) for w in words if w not in ignore_words]
words = sorted(list(set(words)))
# sort classes
classes = sorted(list(set(classes)))
# documents = combination between patterns and intents
print(len(documents), "documents")

47 documents


In [6]:
# classes = intents
print(len(classes), "classes", classes)
# words = all words, vocabulary
print(len(words), "unique lemmatized words", words)


9 classes ['adverse_drug', 'blood_pressure', 'blood_pressure_search', 'goodbye', 'greeting', 'hospital_search', 'options', 'pharmacy_search', 'thanks']
88 unique lemmatized words ["'s", ',', 'a', 'adverse', 'all', 'anyone', 'are', 'awesome', 'be', 'behavior', 'blood', 'by', 'bye', 'can', 'causing', 'chatting', 'check', 'could', 'data', 'day', 'detail', 'do', 'dont', 'drug', 'entry', 'find', 'for', 'give', 'good', 'goodbye', 'have', 'hello', 'help', 'helpful', 'helping', 'hey', 'hi', 'history', 'hola', 'hospital', 'how', 'i', 'id', 'is', 'later', 'list', 'load', 'locate', 'log', 'looking', 'lookup', 'management', 'me', 'module', 'nearby', 'next', 'nice', 'of', 'offered', 'open', 'patient', 'pharmacy', 'pressure', 'provide', 'reaction', 'related', 'result', 'search', 'searching', 'see', 'show', 'suitable', 'support', 'task', 'thank', 'thanks', 'that', 'there', 'till', 'time', 'to', 'transfer', 'up', 'want', 'what', 'which', 'with', 'you']


In [7]:

pickle.dump(words, open('words.pkl', 'wb'))
pickle.dump(classes, open('classes.pkl', 'wb'))


In [8]:

# create our training data
training = []
# create an empty array for our output
output_empty = [0] * len(classes)


In [9]:
# training set, bag of words for each sentence
for doc in documents:
    # initialize our bag of words
    bag = []
    # list of tokenized words for the pattern
    pattern_words = doc[0]
    # lemmatize each word - create base word, in attempt to represent related words
    pattern_words = [lemmatizer.lemmatize(word.lower()) for word in pattern_words]
    # create our bag of words array with 1, if word match found in current pattern
    for w in words:
        bag.append(1) if w in pattern_words else bag.append(0)
    
    # output is a '0' for each tag and '1' for current tag (for each pattern)
    output_row = list(output_empty)
    output_row[classes.index(doc[1])] = 1
    
    training.append([bag, output_row])

In [10]:
training

[[[0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   1,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   1,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0],
  [0, 0, 0, 0, 1, 0, 0, 0, 0]],
 [[0,
   0,
   0,
   0,
   0,
   0,
   1,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   1,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
 

In [11]:

# FIX: Separate features and labels before converting to numpy arrays
train_x = []
train_y = []
for feature, label in training:
    train_x.append(feature)
    train_y.append(label)

In [12]:

# Convert to numpy arrays
train_x = np.array(train_x)
train_y = np.array(train_y)

In [13]:

print("Training data created")
print("Shape of train_x:", train_x.shape)
print("Shape of train_y:", train_y.shape)



Training data created
Shape of train_x: (47, 88)
Shape of train_y: (47, 9)


In [14]:
from sklearn.ensemble import RandomForestClassifier
ran=RandomForestClassifier()
ran.fit(train_x,train_y)

ran.score(train_x,train_y)*100

100.0

In [15]:

# Create model - 3 layers. First layer 128 neurons, second layer 64 neurons and 3rd output layer contains number of neurons
# equal to number of intents to predict output intent with softmax
from keras.layers import BatchNormalization
model = Sequential()
model.add(Dense(5, input_shape=(len(train_x[0]),), activation='relu'))
model.add(Dropout(0.5))
model.add(BatchNormalization())

model.add(Dense(40, activation='relu'))
model.add(BatchNormalization())
model.add(Dense(4, activation='relu'))
model.add(Dense(len(train_y[0]), activation='softmax'))


In [16]:
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 5)                 445       
                                                                 
 dropout (Dropout)           (None, 5)                 0         
                                                                 
 batch_normalization (BatchN  (None, 5)                20        
 ormalization)                                                   
                                                                 
 dense_1 (Dense)             (None, 40)                240       
                                                                 
 batch_normalization_1 (Batc  (None, 40)               160       
 hNormalization)                                                 
                                                                 
 dense_2 (Dense)             (None, 4)                 1

In [17]:

# Compile model. Stochastic gradient descent with Nesterov accelerated gradient gives good results for this model
# Note: For newer versions of Keras/TensorFlow, use learning_rate instead of lr
sgd = SGD(learning_rate=0.01, decay=1e-6, momentum=0.9, nesterov=True)
model.compile(loss='categorical_crossentropy', optimizer=sgd, metrics=['accuracy'])


In [18]:

# Fitting and saving the model 
hist = model.fit(train_x, train_y, epochs=200, batch_size=5, verbose=1)
model.save('chatbot_model.h5')

print("Model created and saved")

Epoch 1/200
10/10 [==============================] - 3s 7ms/step - loss: 2.3307 - accuracy: 0.1064
Epoch 2/200
10/10 [==============================] - 0s 6ms/step - loss: 2.0625 - accuracy: 0.2340
Epoch 3/200
10/10 [==============================] - 0s 7ms/step - loss: 2.0379 - accuracy: 0.2553
Epoch 4/200
10/10 [==============================] - 0s 6ms/step - loss: 2.3423 - accuracy: 0.1277
Epoch 5/200
10/10 [==============================] - 0s 6ms/step - loss: 2.1756 - accuracy: 0.1702
Epoch 6/200
10/10 [==============================] - 0s 6ms/step - loss: 2.1762 - accuracy: 0.0851
Epoch 7/200
10/10 [==============================] - 0s 5ms/step - loss: 2.1349 - accuracy: 0.1489
Epoch 8/200
10/10 [==============================] - 0s 6ms/step - loss: 2.1228 - accuracy: 0.1489
Epoch 9/200
10/10 [==============================] - 0s 6ms/step - loss: 2.0652 - accuracy: 0.2766
Epoch 10/200
10/10 [==============================] - 0s 6ms/step - loss: 2.0392 - accuracy: 0.2340
Epoch 11/